# 🎬 The LoRA Sprint — Reelsense Rank & Module Study
### Rotten Tomatoes sentiment classification, DistilBERT

This notebook is a **scaffold only** — it sets up the pipeline shape (same shape as the class SST-2 lab),
but every result, checkpoint answer, and the final memo are yours to produce by actually running the
cells and reading your own numbers.

**What's in here:** Part A (baseline), Part B (rank sweep r=2,8,32), Part C (target_module ablation).
**What's NOT in here:** the bonus storage question, and the recommendation memo — you're writing those.


## Part A — Setup and Baseline

### Concepts before you run anything

- **`rotten_tomatoes` dataset**: a *different* dataset than the SST-2 one you saw in class. Same task shape
  (binary sentiment), different text distribution (critic reviews vs. movie-review sentences), different
  vocabulary and sentence length. This matters because a LoRA config that works well on SST-2 isn't
  guaranteed to transfer — you're validating the *technique*, not reusing a memorized result.
- **`max_length=64`** instead of 128 (used in class): Rotten Tomatoes critic snippets tend to be shorter
  than SST-2 sentences. Shorter max length → less padding waste → faster training. This is a dataset-specific
  choice, not a rule.
- **Baseline (no fine-tuning)**: DistilBERT's classification head (`pre_classifier` + `classifier`) is
  **randomly initialized** — it's never seen this task. So baseline accuracy should sit close to 50%
  (chance, since this is a 2-class problem). If it doesn't, that's worth noticing, not ignoring.


In [ ]:
!pip install -q "transformers>=4.28.0" "datasets" "peft" "accelerate" "evaluate" "scikit-learn" "pandas" "matplotlib"

import os, time, random, numpy as np, torch, pandas as pd, matplotlib.pyplot as plt
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

BASE_MODEL = "distilbert-base-uncased"
MAX_LEN = 64


**Load Rotten Tomatoes, take a 2,000-example subset (train), keep full validation.**

Note: some mirrors of this dataset live under different repo names on the Hub. If `rotten_tomatoes`
doesn't resolve, try the fully-qualified `cornell-movie-review-data/rotten_tomatoes` — check the dataset
card link in your brief if you hit a `DatasetNotFoundError`.

In [ ]:
raw = load_dataset("rotten_tomatoes")

train_ds = raw["train"].shuffle(seed=SEED).select(range(2000))
valid_ds = raw["validation"]

print(train_ds)
print(valid_ds)
print(train_ds[0])


**Tokenizer, preprocessing, dynamic-padding collator, accuracy function, parameter counter.**

- `DataCollatorWithPadding` pads *within a batch* to the longest sequence in that batch, not to a fixed
  length — more efficient than padding everything to 64 up front.
- `count_params` is the function that will produce every number your rank-sweep and ablation charts depend
  on — it's worth understanding exactly what it counts: total parameters in the model object vs. only the
  ones with `requires_grad=True` (i.e. the ones that will actually receive gradient updates).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(preprocess, batched=True, remove_columns=valid_ds.column_names)
train_tok = train_tok.add_column("labels", train_ds["label"])
valid_tok = valid_tok.add_column("labels", valid_ds["label"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"]}

def count_params(model):
    all_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(all_p), int(train_p), 100.0 * train_p / all_p


**Baseline evaluation — no training at all.**

Record this number (Checkpoint A). This is what a randomly-initialized classification head on top of the
pretrained DistilBERT body gets you for free.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2).to(device)

eval_args = TrainingArguments(
    output_dir="./out_eval_only",
    per_device_eval_batch_size=64 if device == "cuda" else 16,
    report_to="none",
    logging_strategy="no",
)
baseline_trainer = Trainer(model=base_model, args=eval_args, eval_dataset=valid_tok,
                           data_collator=collator, compute_metrics=compute_metrics)

baseline_metrics = baseline_trainer.evaluate()
total_params, _, _ = count_params(base_model)

print(f"Baseline accuracy: {baseline_metrics['eval_accuracy']:.4f}")
print(f"Total params (DistilBERT + classification head): {total_params:,}")

# CHECKPOINT A — record these two numbers before moving on
checkpoint_a = {"baseline_accuracy": baseline_metrics["eval_accuracy"], "total_params": total_params}
checkpoint_a


## Part B — Rank Sweep (r = 2, 8, 32)

### Concepts before you run anything

Recall the LoRA math from class: for a target weight matrix, LoRA learns `ΔW = B·A` where `A` is `r×d`
and `B` is `d×r`, and the forward pass adds `(alpha/r)·B·A·x` to the frozen path.

- **Trainable parameter count scales linearly with `r`**: `params ≈ 2·r·d` per adapted matrix. Going from
  r=2 to r=32 is a 16x increase in adapter parameters, not a small bump — watch for this in your log-scale
  plot.
- **`lora_alpha` scaled with r (alpha = 4r here)**: this keeps the *effective* scaling factor `alpha/r`
  constant (=4) as r changes, so you're isolating the effect of rank/capacity alone, without also changing
  how strongly the adapter's output gets weighted. This is what makes it a fair sweep — one variable at a
  time.
- **What you're actually testing**: does more adapter capacity (higher r) keep buying you accuracy, or does
  it plateau? A plateau tells you the "intrinsic rank" of the update needed for this task is low — i.e. you
  don't need a big adapter to capture it. That's the central evidence for your memo.

The function below is written to be **reused unchanged in Part C** — only `r` and `target_modules` change
between calls.

In [ ]:
def train_lora_and_evaluate(r, target_modules, alpha=None, dropout=0.05,
                             lr=2e-4, epochs=1, tag="lora"):
    """Attaches a fresh LoRA adapter to a fresh DistilBERT, trains it, and returns a results dict."""
    if alpha is None:
        alpha = 4 * r  # common heuristic: alpha = 4 * r, keeps alpha/r ratio constant across the sweep

    lora_base = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2).to(device)

    lora_cfg = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_modules,
        modules_to_save=["pre_classifier", "classifier"],
    )
    lora_model = get_peft_model(lora_base, lora_cfg)

    total_params, trainable_params, trainable_pct = count_params(lora_model)

    args = TrainingArguments(
        output_dir=f"./out_{tag}",
        per_device_train_batch_size=32 if device == "cuda" else 8,
        per_device_eval_batch_size=64 if device == "cuda" else 16,
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        fp16=(device == "cuda"),
        seed=SEED,
    )
    trainer = Trainer(model=lora_model, args=args, train_dataset=train_tok, eval_dataset=valid_tok,
                       data_collator=collator, compute_metrics=compute_metrics)

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    metrics = trainer.evaluate()

    return {
        "config": tag,
        "r": r,
        "target_modules": ",".join(target_modules),
        "eval_accuracy": metrics["eval_accuracy"],
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_pct": trainable_pct,
        "train_time_sec": train_time,
    }


In [ ]:
# --- Rank sweep: r = 2, 8, 32, target_modules fixed at ["q_lin", "v_lin"] ---
rank_sweep_results = []
for r in [2, 8, 32]:
    print(f"\n=== Training LoRA with r={r} ===")
    result = train_lora_and_evaluate(r=r, target_modules=["q_lin", "v_lin"], tag=f"rank_r{r}")
    rank_sweep_results.append(result)
    print(result)

df_rank = pd.DataFrame(rank_sweep_results)
df_rank


**Plot 1 — Accuracy vs. rank.** Does it climb steadily, or flatten out after r=8?

**Plot 2 — Trainable parameters vs. rank (log scale).** This is the cost side of the trade-off you're
plotting accuracy against.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(df_rank["r"], df_rank["eval_accuracy"], marker="o")
ax.set_xscale("log", base=2)
ax.set_xticks(df_rank["r"])
ax.set_xticklabels(df_rank["r"])
ax.set_xlabel("LoRA rank (r)")
ax.set_ylabel("Validation accuracy")
ax.set_title("Accuracy vs. rank (target_modules = q_lin, v_lin)")
for x, y in zip(df_rank["r"], df_rank["eval_accuracy"]):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0,8), ha="center")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6,4))
ax.bar([str(r) for r in df_rank["r"]], df_rank["trainable_params"])
ax.set_yscale("log")
ax.set_xlabel("LoRA rank (r)")
ax.set_ylabel("Trainable parameters (log scale)")
ax.set_title("Trainable parameters vs. rank")
for i, v in enumerate(df_rank["trainable_params"]):
    ax.text(i, v * 1.15, f"{int(v):,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


> **Checkpoint B (write this down yourself):** does accuracy keep improving from r=2 → r=8 → r=32, or
> does it plateau? Where does it plateau, if it does? This is the main evidence your Part D memo will lean on.

## Part C — Target Module Ablation (r fixed = 8)

### Concepts before you run anything

Now you hold `r` constant and vary **which weight matrices get an adapter at all** — this is a different
axis of the same trade-off:

- `["v_lin"]` — smallest adapter: only the value projection gets adapted.
- `["q_lin", "v_lin"]` — what class used: query + value (this is what the original LoRA paper found to be a
  strong default for attention layers).
- `["q_lin", "k_lin", "v_lin", "out_lin"]` — largest adapter: all four attention projections adapted.

Each additional target module roughly doubles the r×d contribution (2·r·d) *per matrix added*, so trainable
parameter count should scale roughly linearly with the *number* of target modules, at fixed r — a different
shape of cost curve than the rank sweep (which was per-matrix, this is across-matrices).

The question you're answering: is adapting *more matrices* a better lever than adapting *fewer matrices
more heavily* (higher r)? Comparing this section to Part B's results directly answers that.

In [ ]:
# --- Target module ablation: r fixed, vary target_modules ---
BEST_R = 8  # replace with your best r from Part B if different

module_configs = [
    ["v_lin"],
    ["q_lin", "v_lin"],
    ["q_lin", "k_lin", "v_lin", "out_lin"],
]

module_ablation_results = []
for modules in module_configs:
    tag = "mods_" + "_".join(modules)
    print(f"\n=== Training LoRA with target_modules={modules} (r={BEST_R}) ===")
    result = train_lora_and_evaluate(r=BEST_R, target_modules=modules, tag=tag)
    module_ablation_results.append(result)
    print(result)

df_modules = pd.DataFrame(module_ablation_results)
df_modules


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
labels = df_modules["target_modules"]
x = np.arange(len(labels))
ax.bar(x, df_modules["eval_accuracy"])
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel("Validation accuracy")
ax.set_title(f"Accuracy vs. target_modules (r={BEST_R})")
for i, v in enumerate(df_modules["eval_accuracy"]):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(x, df_modules["trainable_params"])
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel("Trainable parameters (log scale)")
ax.set_title(f"Trainable parameters vs. target_modules (r={BEST_R})")
for i, v in enumerate(df_modules["trainable_params"]):
    ax.text(i, v * 1.15, f"{int(v):,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


> **Checkpoint C (write this down yourself):** how much does accuracy actually move between the smallest
> adapter (`v_lin` only) and the largest (all four projections)? Is that gain, in your judgment, worth the
> extra trainable parameters?

## Combined results table

Everything you need for the submission form and your memo, in one place.

In [ ]:
all_results = pd.concat([df_rank, df_modules], ignore_index=True)
all_results = all_results[["config", "r", "target_modules", "eval_accuracy",
                            "total_params", "trainable_params", "trainable_pct", "train_time_sec"]]

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
all_results


## Part D — Recommendation Memo

*(Not generated here — this is yours to write, 150–200 words, based on the numbers above.)*

Checklist for what it needs to cover (from the brief):
- The LoRA config you'd standardize on (rank + target_modules), stated plainly
- One sentence quantifying the trade-off (e.g. "r=8 recovers X% of full fine-tune accuracy at Y% of the params")
- A judgment call on where the diminishing-returns point is, and how your data shows it
- One paragraph on whether you'd also recommend QLoRA for the shared base model here, and why (or why not)
  for a model as small as DistilBERT
